In [ ]:
# !pip uninstall -y torchao

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

# Base model + uploaded adapter
BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
ADAPTER_REPO = "poshitha/task2-qwen-qlora-adapter"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto",
    dtype=torch.float16,
)

# Attach LoRA adapter
model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_REPO,
)

# Put model in evaluation mode
model.eval()

# Test prompt
prompt = "Explain QLoRA in simple engineering terms."

# Build chat prompt
messages = [
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

# Tokenize
inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

# Generate response
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=64,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

# Decode only newly generated tokens
generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print("PROMPT:")
print(prompt)

print("\nMODEL RESPONSE:")
print(response.strip())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/59.9M [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


PROMPT:
Explain QLoRA in simple engineering terms.

MODEL RESPONSE:
QLoRA is a method to reduce the memory footprint of large language models (LLMs) while maintaining their performance. In simpler terms, it's like making a big toy car smaller without losing its ability to drive well.

Imagine you have a very large toy car that takes up a lot of space on your shelf.
